In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Llama-70B project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "meta-llama/Llama-3.1-70B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,   # recommended on A100/H100
    device_map="auto"
)


/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|█████████████████████████████| 30/30 [02:55<00:00,  5.85s/it]


In [3]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Llama. You are a helpful medical assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [4]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "gpt5-irr-removed-relevancy-combined-dec-12.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "Llama70B_predictions_on_GPT5_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed row indices from QA_ID (format: "Merge Q123")
    processed_indices = set()
    for qa_id in existing_results['QA_ID']:
        # Extract number from "Merge Q123" -> 123, then convert to 0-indexed (122)
        idx = int(qa_id.split('Q')[1]) - 1
        processed_indices.add(idx)
    
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")


# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
    
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["centaur_question_corr"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        prompt = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Use chat template format (like your working example)
        messages = [
            {"role": "system", "content": "You are Llama. You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # Generate prediction
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Extract only the generated part (not the input)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode the response
        raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Extract the answer letter
        extracted_answer = extract_answer_letter(raw_response)
        
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "Llama70B_predictions_on_GPT5.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Unnamed: 0', 'ID_corr', 'centaur_question_corr', 'answer_corr', 'data_source_corr', 'majority_vote', 'run1_response', 'run2_response', 'run3_response', 'num_responses', 'question_options']
Starting from scratch...
Processing 1300 rows...
Processing row 1/1300...
✅ Processed Merge Q1: Answer = D
Processing row 2/1300...
✅ Processed Merge Q2: Answer = D
Processing row 3/1300...
✅ Processed Merge Q3: Answer = A
Processing row 4/1300...
✅ Processed Merge Q4: Answer = D
Processing row 5/1300...
✅ Processed Merge Q5: Answer = H
Processing row 6/1300...
✅ Processed Merge Q6: Answer = B
Processing row 7/1300...
✅ Processed Merge Q7: Answer = C
Processing row 8/1300...
✅ Processed Merge Q8: Answer = B
Processing row 9/1300...
✅ Processed Merge Q9: Answer = C
Processing row 10/1300...
✅ Processed Merge Q10: Answer = D
Saved progress to CSV after 10 items
Processing row 11/1300...
✅ Processed Merge Q11: Answer = C
Processing row 12/1300...
✅ Processed Merge Q12: Answer = B
P

✅ Processed Merge Q125: Answer = D
Processing row 126/1300...
✅ Processed Merge Q126: Answer = D
Processing row 127/1300...
✅ Processed Merge Q127: Answer = D
Processing row 128/1300...
✅ Processed Merge Q128: Answer = D
Processing row 129/1300...
✅ Processed Merge Q129: Answer = A
Processing row 130/1300...
✅ Processed Merge Q130: Answer = C
Saved progress to CSV after 130 items
Processing row 131/1300...
✅ Processed Merge Q131: Answer = B
Processing row 132/1300...
✅ Processed Merge Q132: Answer = I
Processing row 133/1300...
✅ Processed Merge Q133: Answer = C
Processing row 134/1300...
✅ Processed Merge Q134: Answer = B
Processing row 135/1300...
✅ Processed Merge Q135: Answer = B
Processing row 136/1300...
✅ Processed Merge Q136: Answer = A
Processing row 137/1300...
✅ Processed Merge Q137: Answer = D
Processing row 138/1300...
✅ Processed Merge Q138: Answer = A
Processing row 139/1300...
✅ Processed Merge Q139: Answer = C
Processing row 140/1300...
✅ Processed Merge Q140: Answer =

✅ Processed Merge Q250: Answer = D
Saved progress to CSV after 250 items
Processing row 251/1300...
✅ Processed Merge Q251: Answer = D
Processing row 252/1300...
✅ Processed Merge Q252: Answer = B
Processing row 253/1300...
✅ Processed Merge Q253: Answer = F
Processing row 254/1300...
✅ Processed Merge Q254: Answer = D
Processing row 255/1300...
✅ Processed Merge Q255: Answer = B
Processing row 256/1300...
✅ Processed Merge Q256: Answer = D
Processing row 257/1300...
✅ Processed Merge Q257: Answer = C
Processing row 258/1300...
✅ Processed Merge Q258: Answer = D
Processing row 259/1300...
✅ Processed Merge Q259: Answer = J
Processing row 260/1300...
✅ Processed Merge Q260: Answer = D
Saved progress to CSV after 260 items
Processing row 261/1300...
✅ Processed Merge Q261: Answer = B
Processing row 262/1300...
✅ Processed Merge Q262: Answer = B
Processing row 263/1300...
✅ Processed Merge Q263: Answer = E
Processing row 264/1300...
✅ Processed Merge Q264: Answer = I
Processing row 265/13

✅ Processed Merge Q375: Answer = D
Processing row 376/1300...
✅ Processed Merge Q376: Answer = A
Processing row 377/1300...
✅ Processed Merge Q377: Answer = C
Processing row 378/1300...
✅ Processed Merge Q378: Answer = A
Processing row 379/1300...
✅ Processed Merge Q379: Answer = A
Processing row 380/1300...
✅ Processed Merge Q380: Answer = I
Saved progress to CSV after 380 items
Processing row 381/1300...
✅ Processed Merge Q381: Answer = D
Processing row 382/1300...
✅ Processed Merge Q382: Answer = A
Processing row 383/1300...
✅ Processed Merge Q383: Answer = C
Processing row 384/1300...
✅ Processed Merge Q384: Answer = B
Processing row 385/1300...
✅ Processed Merge Q385: Answer = D
Processing row 386/1300...
✅ Processed Merge Q386: Answer = A
Processing row 387/1300...
✅ Processed Merge Q387: Answer = I
Processing row 388/1300...
✅ Processed Merge Q388: Answer = B
Processing row 389/1300...
✅ Processed Merge Q389: Answer = D
Processing row 390/1300...
✅ Processed Merge Q390: Answer =

✅ Processed Merge Q500: Answer = D
Saved progress to CSV after 500 items
Processing row 501/1300...
✅ Processed Merge Q501: Answer = C
Processing row 502/1300...
✅ Processed Merge Q502: Answer = H
Processing row 503/1300...
✅ Processed Merge Q503: Answer = D
Processing row 504/1300...
✅ Processed Merge Q504: Answer = B
Processing row 505/1300...
✅ Processed Merge Q505: Answer = B
Processing row 506/1300...
✅ Processed Merge Q506: Answer = D
Processing row 507/1300...
✅ Processed Merge Q507: Answer = D
Processing row 508/1300...
✅ Processed Merge Q508: Answer = G
Processing row 509/1300...
✅ Processed Merge Q509: Answer = C
Processing row 510/1300...
✅ Processed Merge Q510: Answer = C
Saved progress to CSV after 510 items
Processing row 511/1300...
✅ Processed Merge Q511: Answer = D
Processing row 512/1300...
✅ Processed Merge Q512: Answer = D
Processing row 513/1300...
✅ Processed Merge Q513: Answer = B
Processing row 514/1300...
✅ Processed Merge Q514: Answer = B
Processing row 515/13

...

✅ Processed Merge Q750: Answer = C
Saved progress to CSV after 750 items
Processing row 751/1300...
✅ Processed Merge Q751: Answer = B
Processing row 752/1300...
✅ Processed Merge Q752: Answer = C
Processing row 753/1300...
✅ Processed Merge Q753: Answer = D
Processing row 754/1300...
✅ Processed Merge Q754: Answer = B
Processing row 755/1300...
✅ Processed Merge Q755: Answer = H
Processing row 756/1300...
✅ Processed Merge Q756: Answer = B
Processing row 757/1300...
✅ Processed Merge Q757: Answer = D
Processing row 758/1300...
✅ Processed Merge Q758: Answer = C
Processing row 759/1300...
✅ Processed Merge Q759: Answer = C
Processing row 760/1300...
✅ Processed Merge Q760: Answer = A
Saved progress to CSV after 760 items
Processing row 761/1300...
✅ Processed Merge Q761: Answer = A
Processing row 762/1300...
✅ Processed Merge Q762: Answer = C
Processing row 763/1300...
✅ Processed Merge Q763: Answer = A
Processing row 764/1300...
✅ Processed Merge Q764: Answer = C
Processing row 765/13

✅ Processed Merge Q875: Answer = A
Processing row 876/1300...
✅ Processed Merge Q876: Answer = C
Processing row 877/1300...
✅ Processed Merge Q877: Answer = A
Processing row 878/1300...
✅ Processed Merge Q878: Answer = C
Processing row 879/1300...
✅ Processed Merge Q879: Answer = D
Processing row 880/1300...
✅ Processed Merge Q880: Answer = B
Saved progress to CSV after 880 items
Processing row 881/1300...
✅ Processed Merge Q881: Answer = B
Processing row 882/1300...
✅ Processed Merge Q882: Answer = D
Processing row 883/1300...
✅ Processed Merge Q883: Answer = F
Processing row 884/1300...
✅ Processed Merge Q884: Answer = C
Processing row 885/1300...
✅ Processed Merge Q885: Answer = C
Processing row 886/1300...
✅ Processed Merge Q886: Answer = A
Processing row 887/1300...
✅ Processed Merge Q887: Answer = D
Processing row 888/1300...
✅ Processed Merge Q888: Answer = C
Processing row 889/1300...
✅ Processed Merge Q889: Answer = B
Processing row 890/1300...
✅ Processed Merge Q890: Answer =

✅ Processed Merge Q1000: Answer = B
Saved progress to CSV after 1000 items
Processing row 1001/1300...
✅ Processed Merge Q1001: Answer = C
Processing row 1002/1300...
✅ Processed Merge Q1002: Answer = D
Processing row 1003/1300...
✅ Processed Merge Q1003: Answer = A
Processing row 1004/1300...
✅ Processed Merge Q1004: Answer = C
Processing row 1005/1300...
✅ Processed Merge Q1005: Answer = I
Processing row 1006/1300...
✅ Processed Merge Q1006: Answer = A
Processing row 1007/1300...
✅ Processed Merge Q1007: Answer = I
Processing row 1008/1300...
✅ Processed Merge Q1008: Answer = B
Processing row 1009/1300...
✅ Processed Merge Q1009: Answer = C
Processing row 1010/1300...
✅ Processed Merge Q1010: Answer = B
Saved progress to CSV after 1010 items
Processing row 1011/1300...
✅ Processed Merge Q1011: Answer = C
Processing row 1012/1300...
✅ Processed Merge Q1012: Answer = E
Processing row 1013/1300...
✅ Processed Merge Q1013: Answer = D
Processing row 1014/1300...
✅ Processed Merge Q1014: A

✅ Processed Merge Q1121: Answer = A
Processing row 1122/1300...
✅ Processed Merge Q1122: Answer = A
Processing row 1123/1300...
✅ Processed Merge Q1123: Answer = I
Processing row 1124/1300...
✅ Processed Merge Q1124: Answer = A
Processing row 1125/1300...
✅ Processed Merge Q1125: Answer = D
Processing row 1126/1300...
✅ Processed Merge Q1126: Answer = H
Processing row 1127/1300...
✅ Processed Merge Q1127: Answer = H
Processing row 1128/1300...
✅ Processed Merge Q1128: Answer = G
Processing row 1129/1300...
✅ Processed Merge Q1129: Answer = D
Processing row 1130/1300...
✅ Processed Merge Q1130: Answer = H
Saved progress to CSV after 1130 items
Processing row 1131/1300...
✅ Processed Merge Q1131: Answer = H
Processing row 1132/1300...
✅ Processed Merge Q1132: Answer = B
Processing row 1133/1300...
✅ Processed Merge Q1133: Answer = B
Processing row 1134/1300...
✅ Processed Merge Q1134: Answer = B
Processing row 1135/1300...
✅ Processed Merge Q1135: Answer = D
Processing row 1136/1300...
✅

✅ Processed Merge Q1242: Answer = B
Processing row 1243/1300...
✅ Processed Merge Q1243: Answer = C
Processing row 1244/1300...
✅ Processed Merge Q1244: Answer = A
Processing row 1245/1300...
✅ Processed Merge Q1245: Answer = C
Processing row 1246/1300...
✅ Processed Merge Q1246: Answer = F
Processing row 1247/1300...
✅ Processed Merge Q1247: Answer = C
Processing row 1248/1300...
✅ Processed Merge Q1248: Answer = B
Processing row 1249/1300...
✅ Processed Merge Q1249: Answer = C
Processing row 1250/1300...
✅ Processed Merge Q1250: Answer = F
Saved progress to CSV after 1250 items
Processing row 1251/1300...
✅ Processed Merge Q1251: Answer = A
Processing row 1252/1300...
✅ Processed Merge Q1252: Answer = A
Processing row 1253/1300...
✅ Processed Merge Q1253: Answer = C
Processing row 1254/1300...
✅ Processed Merge Q1254: Answer = A
Processing row 1255/1300...
✅ Processed Merge Q1255: Answer = A
Processing row 1256/1300...
✅ Processed Merge Q1256: Answer = A
Processing row 1257/1300...
✅

In [7]:
# Ensure QA_ID matches row indices in df
def get_id_corr(qa_id):
    try:
        idx = int(qa_id.split('Q')[1]) - 1  # convert "Merge Q123" -> 122 (0-indexed)
        return df.at[idx, "ID_corr"]
    except:
        return None

# Add the new column
output_df["ID_corr"] = output_df["QA_ID"].apply(get_id_corr)

# Drop the old 'Origin' column if you no longer need it
if "Origin" in output_df.columns:
    output_df = output_df.drop(columns=["Origin"])

# Save updated CSV
output_file = paths.PREDICTIONS / "Llama70B_predictions_on_GPT5.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved predictions with ID_corr to {output_file}")


Saved predictions with ID_corr to Llama70B_predictions_on_GPT5.csv


In [8]:
## import pandas as pd
import re
import numpy as np
from scipy import stats

# Load the model predictions
output_df = pd.read_csv(paths.PREDICTIONS / "Llama70B_predictions_on_GPT5.csv")
df = pd.read_csv(paths.DATA / "gpt5-irr-removed-relevancy-combined-dec-12.csv")

# Merge the two dataframes on ID_corr
merged_df = pd.merge(df, output_df[['ID_corr', 'Extracted_Answer']], on='ID_corr', how='inner')

# Compare answers
merged_df['Llama70B_on_GPT5_Match'] = merged_df['answer_corr'] == merged_df['Extracted_Answer']
merged_df['Llama70B_on_GPT5_Match'] = merged_df['Llama70B_on_GPT5_Match'].map({True: 'TRUE', False: 'FALSE'})

# Exact match as 0/1 for statistics
merged_df['match'] = (merged_df['answer_corr'] == merged_df['Extracted_Answer']).astype(int)

# Overall accuracy statistics
accuracy = merged_df['match'].mean()
std_dev = merged_df['match'].std()
n = len(merged_df)
se = std_dev / np.sqrt(n)
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Category-wise analysis (if data_source_corr exists in df)
if 'data_source_corr' in df.columns:
    merged_df['data_source_corr'] = merged_df['data_source_corr']
    category_stats = merged_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()

    # Calculate 95% CI per category
    ci_lower, ci_upper = [], []
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100

    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()

# Summary table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.6431 (64.31%)
Standard Deviation: 0.4793
95% Confidence Interval: [0.6170, 0.6692]
95% CI (percentage): [61.70%, 66.92%]
Sample Size: 1300

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.694158 0.461159 0.019116     0.656614     0.731702        69.415808  46.115946      65.661383      73.170232
      medbullets    207       0.719807 0.450182 0.031290     0.658117     0.781496        71.980676  45.018223      65.811741      78.149612
        medxpert    318       0.333333 0.472147 0.026477     0.281241     0.385426        33.333333  47.214748      28.124104      38.542563
            mmlu    193       0.917098 0.276450 0.019899     0.877849     0.956348        91.709845  27.645048      87.784905      95.634784


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.6431 (64.31%)

## GPT4o

In [9]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "GPT4o_Both_Rounds.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "Llama70B_predictions_on_GPT4o_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed row indices from QA_ID (format: "Merge Q123")
    processed_indices = set()
    for qa_id in existing_results['QA_ID']:
        # Extract number from "Merge Q123" -> 123, then convert to 0-indexed (122)
        idx = int(qa_id.split('Q')[1]) - 1
        processed_indices.add(idx)
    
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")


# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
    
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["New_Sentences"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        prompt = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Use chat template format (like your working example)
        messages = [
            {"role": "system", "content": "You are Llama. You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # Generate prediction
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Extract only the generated part (not the input)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode the response
        raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Extract the answer letter
        extracted_answer = extract_answer_letter(raw_response)
        
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "Llama70B_predictions_on_GPT4o.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Origin', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'ID_corr', 'sentence_number_corr', 'answer_corr', 'data_source_corr', 'REMOVED_Sentences', 'sentence_number_df3', 'step1_excerpts', 'question_options', 'Filtered_Sentences', 'New_Sentences', 'gpt_direct_prediction', '4k_ID', 'Round 1', 'Round 2', 'Round 1 Letter', 'Round 2 Letter', 'Correct Letter']
Starting from scratch...
Processing 1300 rows...
Processing row 1/1300...
✅ Processed Merge Q1: Answer = D
Processing row 2/1300...
✅ Processed Merge Q2: Answer = D
Processing row 3/1300...
✅ Processed Merge Q3: Answer = A
Processing row 4/1300...
✅ Processed Merge Q4: Answer = D
Processing row 5/1300...
✅ Processed Merge Q5: Answer = H
Processing row 6/1300...
✅ Processed Merge Q6: Answer = B
Processing row 7/1300...
✅ Processed Merge Q7: Answer = C
Processing row 8/1300...
✅ Processed Merge Q8: Answer = B
Processing row 9/1300...

✅ Processed Merge Q121: Answer = B
Processing row 122/1300...
✅ Processed Merge Q122: Answer = D
Processing row 123/1300...
✅ Processed Merge Q123: Answer = C
Processing row 124/1300...
✅ Processed Merge Q124: Answer = J
Processing row 125/1300...
✅ Processed Merge Q125: Answer = D
Processing row 126/1300...
✅ Processed Merge Q126: Answer = D
Processing row 127/1300...
✅ Processed Merge Q127: Answer = D
Processing row 128/1300...
✅ Processed Merge Q128: Answer = D
Processing row 129/1300...
✅ Processed Merge Q129: Answer = A
Processing row 130/1300...
✅ Processed Merge Q130: Answer = C
Saved progress to CSV after 130 items
Processing row 131/1300...
✅ Processed Merge Q131: Answer = B
Processing row 132/1300...
✅ Processed Merge Q132: Answer = I
Processing row 133/1300...
✅ Processed Merge Q133: Answer = C
Processing row 134/1300...
✅ Processed Merge Q134: Answer = B
Processing row 135/1300...
✅ Processed Merge Q135: Answer = B
Processing row 136/1300...
✅ Processed Merge Q136: Answer =

✅ Processed Merge Q246: Answer = A
Processing row 247/1300...
✅ Processed Merge Q247: Answer = B
Processing row 248/1300...
✅ Processed Merge Q248: Answer = D
Processing row 249/1300...
✅ Processed Merge Q249: Answer = A
Processing row 250/1300...
✅ Processed Merge Q250: Answer = D
Saved progress to CSV after 250 items
Processing row 251/1300...
✅ Processed Merge Q251: Answer = D
Processing row 252/1300...
✅ Processed Merge Q252: Answer = B
Processing row 253/1300...
✅ Processed Merge Q253: Answer = F
Processing row 254/1300...
✅ Processed Merge Q254: Answer = D
Processing row 255/1300...
✅ Processed Merge Q255: Answer = A
Processing row 256/1300...
✅ Processed Merge Q256: Answer = D
Processing row 257/1300...
✅ Processed Merge Q257: Answer = C
Processing row 258/1300...
✅ Processed Merge Q258: Answer = D
Processing row 259/1300...
✅ Processed Merge Q259: Answer = J
Processing row 260/1300...
✅ Processed Merge Q260: Answer = D
Saved progress to CSV after 260 items
Processing row 261/13

✅ Processed Merge Q371: Answer = D
Processing row 372/1300...
✅ Processed Merge Q372: Answer = C
Processing row 373/1300...
✅ Processed Merge Q373: Answer = C
Processing row 374/1300...
✅ Processed Merge Q374: Answer = C
Processing row 375/1300...
✅ Processed Merge Q375: Answer = D
Processing row 376/1300...
✅ Processed Merge Q376: Answer = A
Processing row 377/1300...
✅ Processed Merge Q377: Answer = C
Processing row 378/1300...
✅ Processed Merge Q378: Answer = A
Processing row 379/1300...
✅ Processed Merge Q379: Answer = A
Processing row 380/1300...
✅ Processed Merge Q380: Answer = I
Saved progress to CSV after 380 items
Processing row 381/1300...
✅ Processed Merge Q381: Answer = D
Processing row 382/1300...
✅ Processed Merge Q382: Answer = A
Processing row 383/1300...
✅ Processed Merge Q383: Answer = C
Processing row 384/1300...
✅ Processed Merge Q384: Answer = B
Processing row 385/1300...
✅ Processed Merge Q385: Answer = D
Processing row 386/1300...
✅ Processed Merge Q386: Answer =

✅ Processed Merge Q496: Answer = B
Processing row 497/1300...
✅ Processed Merge Q497: Answer = A
Processing row 498/1300...
✅ Processed Merge Q498: Answer = C
Processing row 499/1300...
✅ Processed Merge Q499: Answer = H
Processing row 500/1300...
✅ Processed Merge Q500: Answer = D
Saved progress to CSV after 500 items
Processing row 501/1300...
✅ Processed Merge Q501: Answer = C
Processing row 502/1300...
✅ Processed Merge Q502: Answer = H
Processing row 503/1300...
✅ Processed Merge Q503: Answer = D
Processing row 504/1300...
✅ Processed Merge Q504: Answer = B
Processing row 505/1300...
✅ Processed Merge Q505: Answer = B
Processing row 506/1300...
✅ Processed Merge Q506: Answer = B
Processing row 507/1300...
✅ Processed Merge Q507: Answer = D
Processing row 508/1300...
✅ Processed Merge Q508: Answer = G
Processing row 509/1300...
✅ Processed Merge Q509: Answer = C
Processing row 510/1300...
✅ Processed Merge Q510: Answer = C
Saved progress to CSV after 510 items
Processing row 511/13

...

✅ Processed Merge Q746: Answer = C
Processing row 747/1300...
✅ Processed Merge Q747: Answer = D
Processing row 748/1300...
✅ Processed Merge Q748: Answer = D
Processing row 749/1300...
✅ Processed Merge Q749: Answer = B
Processing row 750/1300...
✅ Processed Merge Q750: Answer = C
Saved progress to CSV after 750 items
Processing row 751/1300...
✅ Processed Merge Q751: Answer = C
Processing row 752/1300...
✅ Processed Merge Q752: Answer = C
Processing row 753/1300...
✅ Processed Merge Q753: Answer = D
Processing row 754/1300...
✅ Processed Merge Q754: Answer = B
Processing row 755/1300...
✅ Processed Merge Q755: Answer = H
Processing row 756/1300...
✅ Processed Merge Q756: Answer = B
Processing row 757/1300...
✅ Processed Merge Q757: Answer = D
Processing row 758/1300...
✅ Processed Merge Q758: Answer = C
Processing row 759/1300...
✅ Processed Merge Q759: Answer = C
Processing row 760/1300...
✅ Processed Merge Q760: Answer = A
Saved progress to CSV after 760 items
Processing row 761/13

✅ Processed Merge Q871: Answer = D
Processing row 872/1300...
✅ Processed Merge Q872: Answer = D
Processing row 873/1300...
✅ Processed Merge Q873: Answer = C
Processing row 874/1300...
✅ Processed Merge Q874: Answer = B
Processing row 875/1300...
✅ Processed Merge Q875: Answer = A
Processing row 876/1300...
✅ Processed Merge Q876: Answer = C
Processing row 877/1300...
✅ Processed Merge Q877: Answer = A
Processing row 878/1300...
✅ Processed Merge Q878: Answer = C
Processing row 879/1300...
✅ Processed Merge Q879: Answer = D
Processing row 880/1300...
✅ Processed Merge Q880: Answer = B
Saved progress to CSV after 880 items
Processing row 881/1300...
✅ Processed Merge Q881: Answer = B
Processing row 882/1300...
✅ Processed Merge Q882: Answer = D
Processing row 883/1300...
✅ Processed Merge Q883: Answer = F
Processing row 884/1300...
✅ Processed Merge Q884: Answer = C
Processing row 885/1300...
✅ Processed Merge Q885: Answer = C
Processing row 886/1300...
✅ Processed Merge Q886: Answer =

✅ Processed Merge Q996: Answer = D
Processing row 997/1300...
✅ Processed Merge Q997: Answer = D
Processing row 998/1300...
✅ Processed Merge Q998: Answer = C
Processing row 999/1300...
✅ Processed Merge Q999: Answer = A
Processing row 1000/1300...
✅ Processed Merge Q1000: Answer = B
Saved progress to CSV after 1000 items
Processing row 1001/1300...
✅ Processed Merge Q1001: Answer = C
Processing row 1002/1300...
✅ Processed Merge Q1002: Answer = D
Processing row 1003/1300...
✅ Processed Merge Q1003: Answer = A
Processing row 1004/1300...
✅ Processed Merge Q1004: Answer = C
Processing row 1005/1300...
✅ Processed Merge Q1005: Answer = I
Processing row 1006/1300...
✅ Processed Merge Q1006: Answer = A
Processing row 1007/1300...
✅ Processed Merge Q1007: Answer = I
Processing row 1008/1300...
✅ Processed Merge Q1008: Answer = B
Processing row 1009/1300...
✅ Processed Merge Q1009: Answer = C
Processing row 1010/1300...
✅ Processed Merge Q1010: Answer = B
Saved progress to CSV after 1010 ite

✅ Processed Merge Q1117: Answer = D
Processing row 1118/1300...
✅ Processed Merge Q1118: Answer = A
Processing row 1119/1300...
✅ Processed Merge Q1119: Answer = F
Processing row 1120/1300...
✅ Processed Merge Q1120: Answer = A
Saved progress to CSV after 1120 items
Processing row 1121/1300...
✅ Processed Merge Q1121: Answer = A
Processing row 1122/1300...
✅ Processed Merge Q1122: Answer = A
Processing row 1123/1300...
✅ Processed Merge Q1123: Answer = I
Processing row 1124/1300...
✅ Processed Merge Q1124: Answer = A
Processing row 1125/1300...
✅ Processed Merge Q1125: Answer = D
Processing row 1126/1300...
✅ Processed Merge Q1126: Answer = H
Processing row 1127/1300...
✅ Processed Merge Q1127: Answer = H
Processing row 1128/1300...
✅ Processed Merge Q1128: Answer = G
Processing row 1129/1300...
✅ Processed Merge Q1129: Answer = D
Processing row 1130/1300...
✅ Processed Merge Q1130: Answer = H
Saved progress to CSV after 1130 items
Processing row 1131/1300...
✅ Processed Merge Q1131: A

✅ Processed Merge Q1238: Answer = D
Processing row 1239/1300...
✅ Processed Merge Q1239: Answer = D
Processing row 1240/1300...
✅ Processed Merge Q1240: Answer = G
Saved progress to CSV after 1240 items
Processing row 1241/1300...
✅ Processed Merge Q1241: Answer = A
Processing row 1242/1300...
✅ Processed Merge Q1242: Answer = B
Processing row 1243/1300...
✅ Processed Merge Q1243: Answer = C
Processing row 1244/1300...
✅ Processed Merge Q1244: Answer = A
Processing row 1245/1300...
✅ Processed Merge Q1245: Answer = C
Processing row 1246/1300...
✅ Processed Merge Q1246: Answer = F
Processing row 1247/1300...
✅ Processed Merge Q1247: Answer = C
Processing row 1248/1300...
✅ Processed Merge Q1248: Answer = B
Processing row 1249/1300...
✅ Processed Merge Q1249: Answer = C
Processing row 1250/1300...
✅ Processed Merge Q1250: Answer = F
Saved progress to CSV after 1250 items
Processing row 1251/1300...
✅ Processed Merge Q1251: Answer = A
Processing row 1252/1300...
✅ Processed Merge Q1252: A

In [11]:
## import pandas as pd
import re
import numpy as np
from scipy import stats

# Load the model predictions
output_df = pd.read_csv(paths.PREDICTIONS / "Llama70B_predictions_on_GPT4o.csv")
df = pd.read_csv(paths.DATA / "GPT4o_Both_Rounds.csv")

# Merge the two dataframes on ID_corr
merged_df = pd.merge(df, output_df[['Origin', 'Extracted_Answer']], on='Origin', how='inner')

# Compare answers
merged_df['Llama70B_on_GPT5_Match'] = merged_df['answer_corr'] == merged_df['Extracted_Answer']
merged_df['Llama70B_on_GPT5_Match'] = merged_df['Llama70B_on_GPT5_Match'].map({True: 'TRUE', False: 'FALSE'})

# Exact match as 0/1 for statistics
merged_df['match'] = (merged_df['answer_corr'] == merged_df['Extracted_Answer']).astype(int)

# Overall accuracy statistics
accuracy = merged_df['match'].mean()
std_dev = merged_df['match'].std()
n = len(merged_df)
se = std_dev / np.sqrt(n)
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Category-wise analysis (if data_source_corr exists in df)
if 'data_source_corr' in df.columns:
    merged_df['data_source_corr'] = merged_df['data_source_corr']
    category_stats = merged_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()

    # Calculate 95% CI per category
    ci_lower, ci_upper = [], []
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100

    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()

# Summary table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.6508 (65.08%)
Standard Deviation: 0.4769
95% Confidence Interval: [0.6248, 0.6767]
95% CI (percentage): [62.48%, 67.67%]
Sample Size: 1300

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.702749 0.457441 0.018962     0.665508     0.739991        70.274914  45.744106      66.550762      73.999066
      medbullets    207       0.714976 0.452520 0.031452     0.652966     0.776986        71.497585  45.252031      65.296610      77.698559
        medxpert    318       0.339623 0.474328 0.026599     0.287290     0.391955        33.962264  47.432753      28.728982      39.195546
            mmlu    193       0.937824 0.242103 0.017427     0.903451     0.972197        93.782383  24.210326      90.345093      97.219674


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.6508 (65.08%)